[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/exercices/seance4_exercices.ipynb)

# Séance 3.4 — Régression linéaire — expliquer, et de combien

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'étude de cas en binôme)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- ajuster une régression avec `smf.ols("y ~ x", donnees).fit()`
- lire un coefficient, sa p-value et son intervalle de confiance
- dire ce que le R² mesure — et ce qu'il ne mesure pas
- interpréter un coefficient « toutes choses égales par ailleurs »
- faire entrer une variable qualitative dans un modèle
- reconnaître une extrapolation et refuser d'y répondre

## Comment ça marche

Chaque exercice se termine par une cellule de **vérification** qui vous dit immédiatement si votre réponse est bonne. Exécutez-la après avoir complété votre code.

Les `____` sont les trous à remplir. Tout le reste est déjà écrit.

> ⚠️ Si la vérification affiche `NameError`, c'est que la cellule au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la, puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")
cli = pd.read_csv(BASE + "clients_ca.csv")

print(cmd.shape, cli.shape)
cli.head(3)

### Le contexte

> **Votre mission :**
> Le comité de direction veut **augmenter le panier moyen** et vous demande une note d'une page : *sur quoi faut-il agir, et de combien peut-on espérer bouger ?* Les huit étapes ci-dessous vous y mènent — la dernière est le livrable. Travaillez en binôme, une tablette pour le code, une pour les notes.

In [ ]:
# Rien a completer ici : executez simplement la cellule
print(len(cmd), "commandes |", len(cli), "clients")

In [ ]:
verifier("0 - donnees pretes", len(cmd) == 1955 and len(cli) == 472,
         "relancez la cellule de preparation")

### Étape 1 — De quoi parle-t-on ? (rappel 3.1)

> **Votre mission :**
> - Sur `cli` (un client par ligne), calculer la moyenne et la médiane du `ca` → `moy_cli` et `med_cli`, arrondies à 2 décimales.
> - Lequel des deux mettriez-vous dans la note ?

In [ ]:
moy_cli = round(cli["ca"].____(), 2)
med_cli = round(cli["ca"].____(), 2)

print("moyenne", moy_cli, "| mediane", med_cli)

In [ ]:
verifier("1a - CA moyen par client", moy_cli == 2442.61, "mean()")
verifier("1b - CA median par client", med_cli == 805.3, "median()")

### Étape 2 — Y a-t-il un lien ? (rappel 3.3)

> **Votre mission :**
> - Calculer la corrélation entre le nombre de commandes (`ncmd`) et le chiffre d'affaires du client → `r_ncmd` (3 décimales).
> - Tracer le nuage de points correspondant.

In [ ]:
r_ncmd = round(cli["ca"].corr(cli["____"]), 3)

cli.plot(kind="scatter", x="ncmd", y="ca", alpha=0.4, figsize=(7, 4))
plt.title("Chiffre d'affaires et nombre de commandes")
plt.show()
print("correlation :", r_ncmd)

In [ ]:
verifier("2 - correlation ncmd / ca", r_ncmd == 0.868,
         "cli['ca'].corr(cli['ncmd'])")

### Étape 3 — De combien ? La première régression

> **Votre mission :**
> - Ajuster `ca ~ ncmd` sur `cli` → `mod1`.
> - Récupérer le coefficient de `ncmd` → `coef_ncmd` (2 décimales) et le R² → `r2_1` (3 décimales).

In [ ]:
mod1 = smf.ols("ca ~ ____", cli).____()

coef_ncmd = round(mod1.params["ncmd"], 2)
r2_1 = round(mod1.rsquared, 3)
print("chaque commande supplementaire :", coef_ncmd, "euros | R2 :", r2_1)

In [ ]:
verifier("3a - coefficient de ncmd", coef_ncmd == 759.74,
         "la formule s'ecrit \"ca ~ ncmd\", et il faut .fit()")
verifier("3b - R2 du modele", r2_1 == 0.754, "mod1.rsquared")

### Étape 4 — Une variable qui n'apporte rien

> **Votre mission :**
> - L'ancienneté du client (`anc`, en jours depuis l'inscription) devrait compter. Vérifions.
> - Ajuster `ca ~ ncmd + anc` → `mod2`. Récupérer la p-value de `anc` → `p_anc` (3 décimales) et le R² → `r2_2`.
> - Comparez `r2_2` à `r2_1`.

In [ ]:
mod2 = smf.ols("ca ~ ncmd + ____", cli).fit()

p_anc = round(mod2.pvalues["anc"], 3)
r2_2 = round(mod2.rsquared, 3)
print("p-value de l'anciennete :", p_anc, "| R2 :", r2_2, "contre", r2_1)

In [ ]:
verifier("4a - p-value de l'anciennete", p_anc == 0.241, "mod2.pvalues['anc']")
verifier("4b - R2 inchange", r2_2 == 0.754,
         "comparez avec r2_1 : la variable ajoutee n'apporte rien")

### Étape 5 — Le coefficient qui change de sens

> **Votre mission :**
> - On revient aux commandes. Ajuster `ca ~ nart` puis `ca ~ nart + qte` sur `cmd`.
> - Relever le coefficient de `nart` dans chacun → `c_seul` et `c_avec_qte` (2 décimales).
> - Comment expliqueriez-vous cet écart au comité ?

In [ ]:
ma = smf.ols("ca ~ nart", cmd).fit()
mb = smf.ols("ca ~ nart + ____", cmd).fit()

c_seul = round(ma.params["nart"], 2)
c_avec_qte = round(mb.params["nart"], 2)
print("nart seul :", c_seul, "| nart avec qte :", c_avec_qte)

In [ ]:
verifier("5a - nart seul", c_seul == 15.93, "modele a une seule variable")
verifier("5b - nart avec qte", c_avec_qte == 2.05,
         "ajoutez qte a la formule, separe par un +")

### Étape 6 — Les marchés, à quantité égale

> **Votre mission :**
> - Sur les quatre pays les plus présents, ajuster `ca ~ qte + pays` → `mod4`.
> - Relever le coefficient du Royaume-Uni → `coef_uk` (2 décimales) et la p-value de l'Irlande → `p_irl` (3 décimales).
> - Souvenez-vous de la séance 3.2 : le panier irlandais y était bien plus élevé.

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

mod4 = smf.ols("ca ~ qte + ____", sub).fit()
coef_uk = round(mod4.params["pays[T.Royaume-Uni]"], 2)
p_irl = round(mod4.pvalues["pays[T.Irlande]"], 3)
print("Royaume-Uni :", coef_uk, "euros | p Irlande :", p_irl)

In [ ]:
verifier("6a - coefficient du Royaume-Uni", coef_uk == -99.59,
         "la formule est \"ca ~ qte + pays\"")
verifier("6b - p-value de l'Irlande", p_irl == 0.346,
         "mod4.pvalues, avec l'etiquette pays[T.Irlande]")

### Étape 7 — Prédire, et savoir s'arrêter

> **Votre mission :**
> - Avec `ma` (le modèle `ca ~ nart`), prédire le montant d'une commande de 30 références → `pred30` (2 décimales).
> - Puis d'une commande de 1000 références → `pred1000`.
> - Laquelle des deux prédictions refuseriez-vous de communiquer, et pourquoi ?

In [ ]:
pred30 = round(ma.predict(pd.DataFrame({"nart": [30]})).iloc[0], 2)
pred1000 = round(ma.predict(pd.DataFrame({"nart": [____]})).iloc[0], 2)

print(pred30, "euros |", pred1000, "euros")
print("maximum observe :", cmd["nart"].max(), "references")

In [ ]:
verifier("7a - prediction a 30 references", pred30 == 699.98,
         "predict() attend un DataFrame avec la meme colonne")
verifier("7b - prediction a 1000 references", pred1000 == 16156.37,
         "meme commande, avec 1000 a la place de 30")

### Étape 8 — Le livrable

> **Votre mission :**
> - Vous avez tout ce qu'il faut. Rédigez la note en commentaire, dans la cellule ci-dessous.
> - **Format imposé :** trois constats chiffrés, puis une recommandation, puis une limite que vous assumez.
> - Contrainte : chaque chiffre annoncé doit venir d'une étape précédente, et être accompagné de ce qui le rend crédible (p-value, effectif ou intervalle).

In [ ]:
# Recapitulatif de vos resultats — executez, puis redigez en dessous
print("CA median par client   :", med_cli)
print("Effet d'une commande   :", coef_ncmd, "euros (R2", r2_1, ")")
print("Effet d'une reference  :", c_avec_qte, "euros a quantite egale")
print("Ecart Royaume-Uni      :", coef_uk, "euros a quantite egale")

# NOTE AU COMITE
# Constat 1 :
# Constat 2 :
# Constat 3 :
# Recommandation :
# Limite :

In [ ]:
verifier("8 - resultats disponibles pour la note",
         med_cli == 805.3 and coef_ncmd == 759.74 and c_avec_qte == 2.05,
         "reprenez les etapes 1, 3 et 5 avant de rediger")